In [4]:
import numpy as np
from scipy.linalg import eig
import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from Unifac import Unifac
from sklearn.metrics import root_mean_squared_error
from UnifacDortmund import UnifacDortmund
from PhaseCalculation import CalculateComps as cc


tie_lines = pl.read_excel(r"C:\Juan\Maestría\Code\oleosim\ExperimentalData\data\No 47exp\refined_lle_data_v5.xlsx", engine="calamine")
group_contribution = pl.read_excel(r"C:\Juan\Maestría\Code\oleosim\ExperimentalData\data\group_contribution.xlsx", engine="calamine")
comps_MW = pl.read_excel(r"C:\Juan\Maestría\Code\oleosim\ExperimentalData\data\comps_MW.xlsx", engine="calamine")

tie_lines = tie_lines.with_columns(
        pl.col("Sub Ref").str.slice(0, 1).alias("Sub Ref")
    ).with_row_index()

tie_lines = tie_lines.drop(["Type Alcohol"])
tie_lines_train = tie_lines.sample(fraction=0.8, with_replacement=False, seed=42).sort("index")
tie_lines_validate = tie_lines.filter(~tie_lines["index"].is_in(tie_lines_train["index"]))
tie_lines_train = tie_lines_train.drop(["index"])
tie_lines_validate = tie_lines_validate.drop(["index"])

group_contribution = group_contribution.drop(["MAG/DAG Profile?", "Additional FAME Profile?", "Acid mix Profile?", "Type Alcohol"])
comps_MW = comps_MW.drop(["Type Alcohol"])
tie_lines = tie_lines.drop(["index"])
group_tie_lines = tie_lines.group_by(["Ref", "Sub Ref", "Temperature (K)"], maintain_order=True)     

tie_lines_eval = tie_lines.select(['FAME/FAEE_L', 'GLYCEROL_L', 'ALCOHOL_L', 'OIL_L', 'DAG_L', 'MAG_L', 'ACID_L', 'WATER_L', 'FAME/FAEE_H', 'GLYCEROL_H', 'ALCOHOL_H', 'OIL_H', 'DAG_H', 'MAG_H', 'ACID_H', 'WATER_H'])
tie_lines_eval_np = tie_lines_eval.to_numpy()    
thermo_model = UnifacDortmund


def calculate_jacobian_numerical(f, x, h=1e-6):
    """
    Calculate the Jacobian matrix numerically using finite differences.
    Using a larger step size for better numerical stability.
    
    Args:
        f: Function that takes x and returns the iteration mapping
        x: Current point (flattened array of compositions)
        h: Step size for finite differences
    
    Returns:
        Jacobian matrix
    """
    n = len(x)
    J = np.zeros((n, n))
    
    try:
        f_x = f(x)
        if len(f_x) != n:
            raise ValueError(f"Function output size {len(f_x)} doesn't match input size {n}")
        
        for i in range(n):
            try:
                x_plus = x.copy()
                x_plus[i] += h
                f_x_plus = f(x_plus)
                
                if len(f_x_plus) == n:
                    J[:, i] = (f_x_plus - f_x) / h
                else:
                    print(f"Warning: Inconsistent function output size at column {i}")
                    J[:, i] = 0  # Default to zero if there's an issue
                    
            except Exception as e:
                print(f"Error calculating Jacobian column {i}: {e}")
                J[:, i] = 0  # Default to zero if there's an issue
                
    except Exception as e:
        print(f"Error in Jacobian calculation: {e}")
        return np.eye(n)  # Return identity matrix as fallback
    
    return J

def analyze_stability(calculate_comps_func, data, v, data_MW, temperature, thermo_model, GIP, 
                     w_light, w_heavy, phi, return_details=False):
    """
    Analyze the stability of the successive substitution algorithm by calculating
    the eigenvalues of the Jacobian matrix.
    
    Args:
        calculate_comps_func: The calculate_comps function
        data: Global/feed compositions
        v: Group contribution matrix
        data_MW: Molecular weights
        temperature: Temperature in K
        thermo_model: Thermodynamic model (UNIFAC/UNIFAC Dortmund)
        GIP: Group interaction parameters
        w_light: Light phase compositions
        w_heavy: Heavy phase compositions  
        phi: Phase fraction
        return_details: Whether to return detailed analysis
    
    Returns:
        spectral_radius: Maximum eigenvalue magnitude
        eigenvalues: All eigenvalues (if return_details=True)
        stability_analysis: Dictionary with stability metrics (if return_details=True)
    """
    
    # Create the phase distribution object
    phase_distribution = thermo_model(GIP, v, data_MW, temperature)
    
    def iteration_mapping(x_flat):
        """
        Define the iteration mapping F(x) where x_{k+1} = F(x_k)
        x_flat contains [w_light_flat, w_heavy_flat] for all tie lines
        """
        try:
            n_tie_lines = data.shape[0]
            n_comps = 8
            
            # Reshape flattened input
            w_light_current = x_flat[:n_tie_lines*n_comps].reshape(n_tie_lines, n_comps)
            w_heavy_current = x_flat[n_tie_lines*n_comps:].reshape(n_tie_lines, n_comps)
            
            # Calculate activity coefficients with error handling
            gamma_light = np.ones((n_tie_lines, n_comps))
            gamma_heavy = np.ones((n_tie_lines, n_comps))
            
            for i in range(n_tie_lines):
                try:
                    gamma_light_single = phase_distribution.calculate_gamma(w_light_current[i])
                    gamma_heavy_single = phase_distribution.calculate_gamma(w_heavy_current[i])
                    
                    # Ensure proper shape
                    if np.isscalar(gamma_light_single):
                        gamma_light[i, :] = gamma_light_single
                    else:
                        gamma_light[i, :] = np.atleast_1d(gamma_light_single)[:n_comps]
                        
                    if np.isscalar(gamma_heavy_single):
                        gamma_heavy[i, :] = gamma_heavy_single
                    else:
                        gamma_heavy[i, :] = np.atleast_1d(gamma_heavy_single)[:n_comps]
                        
                except Exception as e:
                    # Use default values if calculation fails
                    gamma_light[i, :] = 1.0
                    gamma_heavy[i, :] = 1.0
            
            # Handle zero compositions - ensure masks have correct shape
            mask_light = data == 0
            mask_heavy = data == 0
            
            # Apply masks only if shapes match
            if mask_light.shape == gamma_light.shape:
                gamma_light[mask_light] = 1
            if mask_heavy.shape == gamma_heavy.shape:
                gamma_heavy[mask_heavy] = 1
                
            # Prevent division by zero
            gamma_light = np.where(gamma_light == 0, 1e-20, gamma_light)
            gamma_heavy = np.where(gamma_heavy == 0, 1e-20, gamma_heavy)
            
            # Calculate K-values
            Ki = np.clip(gamma_light / gamma_heavy, 1e-6, 1e6)
            
            # Solve Rachford-Rice equation (simplified for stability analysis)
            phi_current = np.full((n_tie_lines,), 0.5)
            
            # Calculate new compositions
            phi_reshaped = phi_current.reshape(n_tie_lines, 1)
            w_light_new = data / (1 + (Ki - 1) * phi_reshaped)
            w_heavy_new = Ki * w_light_new
            
            # Apply constraints
            w_light_new = np.clip(w_light_new, 1e-10, 0.999)
            w_heavy_new = np.clip(w_heavy_new, 1e-10, 0.999)
            
            # Phase identification logic with bounds checking
            if w_light_new.shape[1] >= 8 and w_heavy_new.shape[1] >= 8:
                condition_GLY = w_light_new[:, 1] > w_heavy_new[:, 1]
                condition_OIL_1 = w_light_new[:, 3] > w_heavy_new[:, 3]
                condition_OIL_2 = w_light_new[:, 3] > w_light_new[:, 7]
                condition_OIL_3 = w_light_new[:, 7] > w_heavy_new[:, 7]
                combined_OIL_1 = condition_OIL_1 & condition_OIL_2 & condition_OIL_3
                condition_WATER = w_light_new[:, 7] > w_heavy_new[:, 7]
                total_condition = condition_GLY | combined_OIL_1 | condition_WATER
                
                # Swap phases if needed
                temp_light = w_light_new[total_condition].copy()
                temp_heavy = w_heavy_new[total_condition].copy()
                w_light_new[total_condition] = temp_heavy
                w_heavy_new[total_condition] = temp_light
            
            # Return flattened result
            return np.concatenate([w_light_new.flatten(), w_heavy_new.flatten()])
            
        except Exception as e:
            # Return current state if anything fails
            print(f"Error in iteration_mapping: {e}")
            return x_flat
    
    # Current state vector with validation
    try:
        x_current = np.concatenate([w_light.flatten(), w_heavy.flatten()])
        if np.any(np.isnan(x_current)) or np.any(np.isinf(x_current)):
            raise ValueError("Invalid values (NaN or Inf) in current state")
    except Exception as e:
        print(f"Error creating state vector: {e}")
        return 1.0  # Return stable value as fallback
    
    # Calculate Jacobian matrix with error handling
    try:
        J = calculate_jacobian_numerical(iteration_mapping, x_current)
        if J.shape[0] != J.shape[1] or J.shape[0] != len(x_current):
            raise ValueError(f"Jacobian shape {J.shape} inconsistent with state size {len(x_current)}")
    except Exception as e:
        print(f"Error calculating Jacobian: {e}")
        return 1.0  # Return stable value as fallback
    
    # Calculate eigenvalues with error handling
    try:
        eigenvalues, eigenvectors = eig(J)
        if len(eigenvalues) == 0:
            raise ValueError("No eigenvalues computed")
    except Exception as e:
        print(f"Error calculating eigenvalues: {e}")
        return 1.0  # Return stable value as fallback
    
    # Calculate spectral radius
    spectral_radius = np.max(np.abs(eigenvalues))
    
    if return_details:
        # Additional stability analysis
        stability_analysis = {
            'spectral_radius': spectral_radius,
            'is_stable': spectral_radius < 1.0,
            'convergence_rate': -np.log(spectral_radius) if spectral_radius > 0 else np.inf,
            'dominant_eigenvalue': eigenvalues[np.argmax(np.abs(eigenvalues))],
            'condition_number': np.linalg.cond(J),
            'n_eigenvalues_outside_unit_circle': np.sum(np.abs(eigenvalues) >= 1.0),
            'eigenvalue_magnitudes': np.abs(eigenvalues)
        }
        
        return spectral_radius, eigenvalues, stability_analysis
    
    return spectral_radius

def modified_calculate_comps_with_stability(data, v, data_MW, temperature, thermo_model, 
                                          GIP, initial_guess=None, analyze_stability_flag=True):
    """
    Modified version of calculate_comps that includes stability analysis.
    
    This function performs the same equilibrium calculation but adds stability monitoring.
    """
    comp_results = np.zeros((data.shape[0], 16))
    phase_distribution = thermo_model(GIP, v, data_MW, temperature)
    mask_light = data == 0
    mask_heavy = data == 0
    
    # Initialize compositions
    if initial_guess is not None:
        initial_w_light_guess = initial_guess[:, 0:8]
        initial_w_heavy_guess = initial_guess[:, 8:16]
        phi_guess = (1 - initial_guess[:, -1])
    else:
        initial_w_light_guess = np.random.random((data.shape[0], 8))
        initial_w_heavy_guess = np.random.random((data.shape[0], 8))
        initial_w_light_guess[mask_light] = 0
        initial_w_heavy_guess[mask_heavy] = 0
        initial_w_light_guess = initial_w_light_guess / initial_w_light_guess.sum(axis=1, keepdims=True)
        initial_w_heavy_guess = initial_w_heavy_guess / initial_w_heavy_guess.sum(axis=1, keepdims=True)
        phi_guess = np.full((data.shape[0],), 0.5)
    
    iter = 0
    relaxation_factor = 0.63
    stability_history = []
    
    while True:
        iter += 1
        
        # Standard successive substitution iteration
        gamma_light = np.apply_along_axis(phase_distribution.calculate_gamma, 1, initial_w_light_guess).reshape(data.shape[0], 8)
        gamma_heavy = np.apply_along_axis(phase_distribution.calculate_gamma, 1, initial_w_heavy_guess).reshape(data.shape[0], 8)
        gamma_light[mask_light] = 1
        gamma_heavy[mask_heavy] = 1
        gamma_light = np.where(gamma_light == 0, 1e-20, gamma_light)
        gamma_heavy = np.where(gamma_heavy == 0, 1e-20, gamma_heavy)
        Ki = np.clip(gamma_light / gamma_heavy, 1e-6, 1e6)
        
        # Solve Rachford-Rice (simplified version)
        phi_current = np.full((data.shape[0],), 0.5)
        sol = phi_current.reshape(data.shape[0], 1)
        
        calculated_w_light = data / (1 + (Ki - 1) * sol)
        calculated_w_heavy = Ki * calculated_w_light
        
        calculated_w_light = np.where(calculated_w_light >= 1, 0.999, calculated_w_light)
        calculated_w_heavy = np.where(calculated_w_heavy >= 1, 0.999, calculated_w_heavy)
        
        # Phase identification
        condition_GLY = calculated_w_light[:, 1] > calculated_w_heavy[:, 1]
        condition_OIL_1 = calculated_w_light[:, 3] > calculated_w_heavy[:, 3]
        condition_OIL_2 = calculated_w_light[:, 3] > calculated_w_light[:, 7]
        condition_OIL_3 = calculated_w_light[:, 7] > calculated_w_heavy[:, 7]
        combined_OIL_1 = condition_OIL_1 & condition_OIL_2 & condition_OIL_3
        condition_WATER = calculated_w_light[:, 7] > calculated_w_heavy[:, 7]
        total_condition = condition_GLY | combined_OIL_1 | condition_WATER
        
        calculated_w_light[total_condition], calculated_w_heavy[total_condition] = (
            calculated_w_heavy[total_condition], calculated_w_light[total_condition])
        
        # Stability analysis
        if analyze_stability_flag and iter % 10 == 0:  # Analyze every 10 iterations to reduce computational cost
            try:
                # Only perform stability analysis if arrays have consistent shapes
                if (initial_w_light_guess.shape == calculated_w_light.shape and 
                    initial_w_heavy_guess.shape == calculated_w_heavy.shape):
                    
                    spectral_radius = analyze_stability(
                        None, data, v, data_MW, temperature, thermo_model, GIP,
                        initial_w_light_guess, initial_w_heavy_guess, phi_guess
                    )
                    stability_history.append((iter, spectral_radius))
                    print(f"Iteration {iter}: Spectral radius = {spectral_radius:.6f}")
                else:
                    print(f"Iteration {iter}: Skipping stability analysis due to shape mismatch")
            except Exception as e:
                print(f"Stability analysis failed at iteration {iter}: {str(e)[:100]}...")
                # Continue without stability analysis
        
        # Check convergence
        light_error = np.absolute(calculated_w_light - initial_w_light_guess)
        heavy_error = np.absolute(calculated_w_heavy - initial_w_heavy_guess)
        
        if (light_error < 0.001).all() & (heavy_error < 0.001).all() or iter >= 100:
            comp_results[:, 0:8] = calculated_w_light
            comp_results[:, 8:16] = calculated_w_heavy
            break
        else:
            initial_w_light_guess = initial_w_light_guess + relaxation_factor * (calculated_w_light - initial_w_light_guess)
            initial_w_heavy_guess = initial_w_heavy_guess + relaxation_factor * (calculated_w_heavy - initial_w_heavy_guess)
    
    # Final stability analysis
    if analyze_stability_flag:
        try:
            final_spectral_radius, eigenvalues, stability_analysis = analyze_stability(
                None, data, v, data_MW, temperature, thermo_model, GIP,
                comp_results[:, 0:8], comp_results[:, 8:16], phi_guess, return_details=True
            )
            
            print(f"\nFinal Stability Analysis:")
            print(f"  Spectral radius: {final_spectral_radius:.6f}")
            print(f"  Stable: {stability_analysis['is_stable']}")
            print(f"  Convergence rate: {stability_analysis['convergence_rate']:.6f}")
            print(f"  Eigenvalues outside unit circle: {stability_analysis['n_eigenvalues_outside_unit_circle']}")
            
            return comp_results, stability_history, stability_analysis
            
        except Exception as e:
            print(f"Final stability analysis failed: {e}")
            return comp_results, stability_history, None
    
    return comp_results

def plot_stability_analysis(stability_history, eigenvalues=None):
    """
    Plot stability analysis results.
    """
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    # Plot spectral radius evolution
    if stability_history:
        iterations, spectral_radii = zip(*stability_history)
        axes[0].plot(iterations, spectral_radii, 'b-o', markersize=4)
        axes[0].axhline(y=1.0, color='r', linestyle='--', label='Stability threshold')
        axes[0].set_xlabel('Iteration')
        axes[0].set_ylabel('Spectral Radius')
        axes[0].set_title('Spectral Radius Evolution')
        axes[0].legend()
        axes[0].grid(True, alpha=0.3)
    
    # Plot eigenvalues in complex plane
    if eigenvalues is not None:
        real_parts = eigenvalues.real
        imag_parts = eigenvalues.imag
        
        # Plot unit circle
        theta = np.linspace(0, 2*np.pi, 100)
        axes[1].plot(np.cos(theta), np.sin(theta), 'r--', label='Unit circle')
        
        # Plot eigenvalues
        axes[1].scatter(real_parts, imag_parts, c='blue', alpha=0.7, s=50)
        axes[1].set_xlabel('Real Part')
        axes[1].set_ylabel('Imaginary Part')
        axes[1].set_title('Eigenvalues in Complex Plane')
        axes[1].legend()
        axes[1].grid(True, alpha=0.3)
        axes[1].axis('equal')
    
    plt.tight_layout()
    plt.show()


# Example usage function
def analyze_your_system_stability(params):
    """
    Modified version of your predict_comps_eval function with stability analysis.
    """
    A = params[0:30]
    B = params[30:60]
    C = params[60:90]
    
    # Assuming you have these variables defined in your main code
    # GIP = cc.define_GIP(A, B, C)
    
    stability_results = []
    
    # This would replace your main loop
    # for name, data in group_tie_lines:
    #     tie_lines_np = data.drop(["Ref", "Sub Ref", "Temperature (K)"]).to_numpy()
    #     v = group_contribution.filter((pl.col("Ref") == name[0]) & (pl.col("Sub Ref") == name[1])).drop(["Ref", "Sub Ref"]).to_numpy()
    #     experiment_MW = comps_MW.filter((pl.col("Ref") == name[0]) & (pl.col("Sub Ref") == name[1])).drop(["Ref", "Sub Ref"]).to_numpy()
    #     
    #     # Use the modified function with stability analysis
    #     results, stability_history, stability_analysis = modified_calculate_comps_with_stability(
    #         tie_lines_np[:,0:8], v, experiment_MW, name[2], thermo_model, GIP, 
    #         initial_guess=tie_lines_np[:,8:25], analyze_stability_flag=True
    #     )
    #     
    #     stability_results.append({
    #         'name': name,
    #         'results': results,
    #         'stability_history': stability_history,
    #         'stability_analysis': stability_analysis
    #     })
    
    return stability_results

# Instructions for integration
print("""
To integrate this stability analysis into your existing code:

1. Replace your cc.calculate_comps calls with modified_calculate_comps_with_stability
2. The function will print stability information during iteration
3. Use analyze_stability function for detailed eigenvalue analysis at any point
4. Use plot_stability_analysis to visualize the stability evolution

Key stability indicators:
- Spectral radius < 1.0: Algorithm will converge
- Spectral radius ≥ 1.0: Algorithm may not converge or converge slowly
- Convergence rate: Higher values indicate faster convergence
- Eigenvalues outside unit circle: Number of unstable modes

The relaxation factor (0.63 in your code) can be adjusted based on stability analysis.
If spectral radius > 1, try reducing the relaxation factor.
""") 


To integrate this stability analysis into your existing code:

1. Replace your cc.calculate_comps calls with modified_calculate_comps_with_stability
2. The function will print stability information during iteration
3. Use analyze_stability function for detailed eigenvalue analysis at any point
4. Use plot_stability_analysis to visualize the stability evolution

Key stability indicators:
- Spectral radius < 1.0: Algorithm will converge
- Spectral radius ≥ 1.0: Algorithm may not converge or converge slowly
- Convergence rate: Higher values indicate faster convergence
- Eigenvalues outside unit circle: Number of unstable modes

The relaxation factor (0.63 in your code) can be adjusted based on stability analysis.
If spectral radius > 1, try reducing the relaxation factor.



In [5]:
def predict_comps_eval(params):

    #A = np.array([-3214.25, -1718.09, -3079.37, 1295.55, 1249.74, 2900.56, 2900.56, 2900.56, -388.62, 259.13, 4925.67, 4925.67, 4925.67, 1047.08, -2424.84, 
                  #2544.80, 2544.80, 2544.80, 258.51, -3227.03, 3714.86, 3714.86, 3714.86, -987.18, -688.85, 20930.91, 5084.03, -1210.44, -4618.51, -14448.32]) 
    
    #B = params[0:30]
    #C = params[30:60]
    
    A = params[0:30]
    B = params[30:60]
    C = params[60:90]
    
    GIP = cc.define_GIP(A, B, C)
    
    results_list = []
    for name, data in group_tie_lines:
        tie_lines_np = data.drop(["Ref", "Sub Ref", "Temperature (K)"]).to_numpy()
        v = group_contribution.filter((pl.col("Ref") == name[0]) & (pl.col("Sub Ref") == name[1])).drop(["Ref", "Sub Ref"]).to_numpy()
        experiment_MW = comps_MW.filter((pl.col("Ref") == name[0]) & (pl.col("Sub Ref") == name[1])).drop(["Ref", "Sub Ref"]).to_numpy()
        experiment_MW.reshape(8) 
        results_list.append(modified_calculate_comps_with_stability(tie_lines_np[:,0:8], v, experiment_MW, name[2], thermo_model, GIP, initial_guess=tie_lines_np[:,8:25])) #tie_lines_np[:,8:25]
    
    results_total = np.vstack(results_list) 
    mask = tie_lines_eval_np == 0
    results_total[mask] = 0 
    
    pl.DataFrame(results_total, schema=['FAME/FAEE_L', 'GLYCEROL_L', 'ALCOHOL_L', 'OIL_L', 'DAG_L', 'MAG_L', 'ACID_L', 'WATER_L', 'FAME/FAEE_H', 'GLYCEROL_H', 'ALCOHOL_H', 'OIL_H', 'DAG_H', 'MAG_H', 'ACID_H', 'WATER_H']).write_excel("results_total_data_damping.xlsx") 
    #pl.DataFrame(tie_lines_eval_np, schema=['FAME/FAEE_L', 'GLYCEROL_L', 'ALCOHOL_L', 'OIL_L', 'DAG_L', 'MAG_L', 'ACID_L', 'WATER_L', 'FAME/FAEE_H', 'GLYCEROL_H', 'ALCOHOL_H', 'OIL_H', 'DAG_H', 'MAG_H', 'ACID_H', 'WATER_H']).write_excel("total_data_damping.xlsx") 
    return root_mean_squared_error(tie_lines_eval_np, results_total)


In [6]:
params = np.loadtxt(r"C:\Juan\Maestría\Code\oleosim\Evaluation\Stochastic_optimization\GA\UDortmundParamsGARun1.out")
#params = np.zeros(60)
print(predict_comps_eval(params))


Final Stability Analysis:
  Spectral radius: 0.000016
  Stable: True
  Convergence rate: 11.017167
  Eigenvalues outside unit circle: 0


C:\Users\USER\AppData\Local\Temp\ipykernel_42016\551920071.py:284: RuntimeWarning: overflow encountered in divide
  Ki = np.clip(gamma_light / gamma_heavy, 1e-6, 1e6)



Final Stability Analysis:
  Spectral radius: 0.000000
  Stable: True
  Convergence rate: inf
  Eigenvalues outside unit circle: 0

Final Stability Analysis:
  Spectral radius: 0.000000
  Stable: True
  Convergence rate: inf
  Eigenvalues outside unit circle: 0

Final Stability Analysis:
  Spectral radius: 0.000000
  Stable: True
  Convergence rate: inf
  Eigenvalues outside unit circle: 0

Final Stability Analysis:
  Spectral radius: 0.000000
  Stable: True
  Convergence rate: inf
  Eigenvalues outside unit circle: 0

Final Stability Analysis:
  Spectral radius: 0.000000
  Stable: True
  Convergence rate: inf
  Eigenvalues outside unit circle: 0

Final Stability Analysis:
  Spectral radius: 0.000000
  Stable: True
  Convergence rate: inf
  Eigenvalues outside unit circle: 0

Final Stability Analysis:
  Spectral radius: 0.000025
  Stable: True
  Convergence rate: 10.610916
  Eigenvalues outside unit circle: 0

Final Stability Analysis:
  Spectral radius: 0.000024
  Stable: True
  Conve

ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (3,) + inhomogeneous part.